# Function 3: Customer Repeat-Purchase Machine Learning Modeling

This notebook trains, validates, and evaluates leakage-safe baseline propensity models using the prepared customer snapshot (`ml_data/ml_customer_dataset.csv`).

- **Target (`repeat_purchase_next_180d`):** `1` if a customer with at least one delivered order before **2018-03-01** placed at least one delivered order between **2018-03-01** and **2018-08-27** (end-exclusive at `2018-08-28`); otherwise `0`.
- **Important Business Note:** This is a **propensity model**, not an uplift model. It ranks customers by likelihood to buy again without proving that a promotional discount will cause incremental purchases. Validate with a randomized holdout experiment before launching offers.


## 1. Imports and Paths

In [1]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    precision_recall_curve,
    roc_curve,
)

warnings.filterwarnings("ignore", category=FutureWarning)

DATA_PATH = Path("ml_data/ml_customer_dataset.csv")
OUTPUT_DIR = Path("model_artifacts")
FIG_DIR = OUTPUT_DIR
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {DATA_PATH}. Run `python build_ml_dataset.py` from the project root first."
    )

df = pd.read_csv(DATA_PATH)
print(f"Dataset loaded: {df.shape[0]:,} rows x {df.shape[1]} columns")
display(df.head())


Dataset loaded: 52,013 rows x 32 columns


,customer_unique_id,historical_orders,historical_spend,avg_order_value,median_order_value,max_order_value,total_items,avg_items_per_order,total_freight,avg_freight_per_order,unique_products,unique_sellers,avg_category_diversity,avg_payment_installments,avg_delivery_days,avg_delivery_delay_days,late_delivery_rate,avg_review_score,review_count,customer_state,customer_city,recency_days,customer_tenure_days,purchase_span_days,orders_per_30_days,used_payment_boleto,used_payment_credit_card,used_payment_debit_card,used_payment_voucher,repeat_purchase_next_180d,snapshot_date,label_window_end
0,0000f46a3911fa3c0805444483337064,1,86.22,86.22,86.22,86.22,1,1.0,17.22,17.22,1,1,1.0,8.0,25.731759,-1.389734,0.0,3.0,1,SC,sao jose,355,355,0,0.084507,0,1,0,0,0,2018-03-01,2018-08-28
1,0000f6ccb0745a6a4b88665a16c9f078,1,43.62,43.62,43.62,43.62,1,1.0,17.63,17.63,1,1,1.0,4.0,20.037083,-11.108970,0.0,4.0,1,PA,belem,139,139,0,0.215827,0,1,0,0,0,2018-03-01,2018-08-28
2,0004aac84e0df4da2b147fca70cf8255,1,196.89,196.89,196.89,196.89,1,1.0,16.89,16.89,1,1,1.0,6.0,13.141134,-7.035463,0.0,5.0,1,SP,sorocaba,106,106,0,0.283019,0,1,0,0,0,2018-03-01,2018-08-28
3,0005e1862207bf6ccc02e4228effd9a0,1,150.12,150.12,150.12,150.12,1,1.0,15.12,15.12,1,1,1.0,3.0,4.375648,-27.643657,0.0,4.0,1,RJ,teresopolis,361,361,0,0.083102,0,1,0,0,0,2018-03-01,2018-08-28
4,0006fdc98a402fceb4eb0ee528f6a8d4,1,29.00,29.00,29.00,29.00,1,1.0,15.10,15.10,1,1,1.0,2.0,16.388646,-11.220266,0.0,3.0,1,ES,mimoso do sul,225,225,0,0.133333,0,1,0,0,0,2018-03-01,2018-08-28


## 2. Data Validation & Integrity Checks

Verify that each row corresponds to a unique `customer_unique_id`, target labels are clean and binary, and metadata/identifier columns are strictly excluded from features.

In [2]:
TARGET = "repeat_purchase_next_180d"
ID_COL = "customer_unique_id"
META_COLS = ["snapshot_date", "label_window_end", "customer_city"]

assert TARGET in df.columns, f"Missing target column: {TARGET}"
assert ID_COL in df.columns, f"Missing identifier column: {ID_COL}"
assert df[ID_COL].is_unique, "Expected exactly one row per customer_unique_id."
assert df[TARGET].notna().all(), "Target contains missing values."
assert set(df[TARGET].unique()).issubset({0, 1}), "Target must be binary (0/1)."
assert df["snapshot_date"].nunique() == 1, "Expected a single snapshot date."
assert df["label_window_end"].nunique() == 1, "Expected a single label window end date."

payment_flag_cols = [c for c in df.columns if c.startswith("used_payment_")]
for col in payment_flag_cols:
    assert set(df[col].unique()).issubset({0, 1}), f"{col} must be binary 0/1."

print("Duplicate customer IDs:", int(df[ID_COL].duplicated().sum()))
print("Snapshot date:", df["snapshot_date"].iloc[0], "| Label window end:", df["label_window_end"].iloc[0])
print(f"Positive repeat buyers: {int(df[TARGET].sum()):,} / {len(df):,} ({df[TARGET].mean() * 100:.3f}%)")

missing_summary = df.isna().sum()
missing_summary = missing_summary[missing_summary > 0]
print("\nColumns with missing values (handled inside Pipeline):")
print(missing_summary if len(missing_summary) > 0 else "None")

Duplicate customer IDs: 0
Snapshot date: 2018-03-01 | Label window end: 2018-08-28
Positive repeat buyers: 593 / 52,013 (1.140%)

Columns with missing values (handled inside Pipeline):
avg_payment_installments      1
avg_review_score            664
dtype: int64


## 3. Feature Preparation & Leakage-Free Data Splitting

We split the dataset using a stratified `80% / 20%` holdout (`X_train`, `X_test`). Inside `X_train`, we create an additional stratified `80% / 20%` split (`X_fit`, `X_val`) so that **model selection** and **decision threshold tuning** happen strictly on `X_val` without ever touching `X_test`.

In [3]:
excluded = [ID_COL, TARGET] + META_COLS
feature_cols = [c for c in df.columns if c not in excluded]

categorical_cols = [c for c in ["customer_state"] if c in feature_cols]
numeric_cols = [c for c in feature_cols if c not in categorical_cols]

print(f"Total features: {len(feature_cols)} ({len(numeric_cols)} numeric, {len(categorical_cols)} categorical)")

X = df[feature_cols].copy()
y = df[TARGET].astype(int)

# Hold-out 20% test set (untouched until final evaluation)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Internal 80/20 split of training data for model & threshold selection
X_fit, X_val, y_fit, y_val = train_test_split(
    X_train, y_train, test_size=0.20, random_state=42, stratify=y_train
)

def make_preprocessor() -> ColumnTransformer:
    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])
    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])
    return ColumnTransformer([
        ("numeric", numeric_pipeline, numeric_cols),
        ("categorical", categorical_pipeline, categorical_cols),
    ])

print(f"Fit rows: {len(X_fit):,} | Val rows: {len(X_val):,} | Test rows: {len(X_test):,}")
print(f"Positive rates -> Fit: {y_fit.mean():.4f} | Val: {y_val.mean():.4f} | Test: {y_test.mean():.4f}")

Total features: 27 (26 numeric, 1 categorical)
Fit rows: 33,288 | Val rows: 8,322 | Test rows: 10,403
Positive rates -> Fit: 0.0114 | Val: 0.0114 | Test: 0.0114


## 4. Baseline Model Comparison (Validation Set Selection)

Because the positive repeat-purchase rate is ~`1.14%`, accuracy is uninformative. We compare candidate models on the validation split (`X_val`) using **Average Precision (PR-AUC)** as the primary selection metric and **ROC-AUC** as a secondary ranking metric.

In [4]:
def get_candidate_models() -> dict:
    return {
        "LogisticRegression_balanced": LogisticRegression(
            class_weight="balanced",
            max_iter=1000,
            solver="liblinear",
            random_state=42,
        ),
        "RandomForest_balanced": RandomForestClassifier(
            n_estimators=150,
            min_samples_leaf=5,
            class_weight="balanced_subsample",
            n_jobs=-1,
            random_state=42,
            max_features="sqrt",
        ),
    }

val_pipes = {}
validation_rows = []

for model_name, estimator in get_candidate_models().items():
    pipe = Pipeline([
        ("preprocess", make_preprocessor()),
        ("model", estimator),
    ])
    pipe.fit(X_fit, y_fit)
    val_scores = pipe.predict_proba(X_val)[:, 1]

    validation_rows.append({
        "model": model_name,
        "validation_pr_auc": average_precision_score(y_val, val_scores),
        "validation_roc_auc": roc_auc_score(y_val, val_scores),
        "val_positive_rate": y_val.mean(),
    })
    val_pipes[model_name] = pipe

validation_metrics = (
    pd.DataFrame(validation_rows)
    .sort_values("validation_pr_auc", ascending=False)
    .reset_index(drop=True)
)
best_name = validation_metrics.loc[0, "model"]

print("Validation Set Comparison:")
display(validation_metrics.round(4))
print(f"Selected best model (by validation PR-AUC): {best_name}")

Validation Set Comparison:
Selected best model (by validation PR-AUC): LogisticRegression_balanced


,model,validation_pr_auc,validation_roc_auc,val_positive_rate
0,LogisticRegression_balanced,0.0427,0.6330,0.0114
1,RandomForest_balanced,0.0294,0.5511,0.0114


## 5. Decision Threshold Tuning (Validation Set Only)

Using **only** the validation model fitted on `X_fit` and evaluated on `X_val`, we sweep probability thresholds to pick the operating point maximizing F1-score without leaking `X_train` or `X_test` labels.

In [5]:
val_selected_pipe = val_pipes[best_name]
val_scores = val_selected_pipe.predict_proba(X_val)[:, 1]

thresholds = [0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60]
threshold_rows = []

for th in thresholds:
    preds = (val_scores >= th).astype(int)
    threshold_rows.append({
        "threshold": th,
        "predicted_customers": int(preds.sum()),
        "precision": precision_score(y_val, preds, zero_division=0),
        "recall": recall_score(y_val, preds, zero_division=0),
        "f1": f1_score(y_val, preds, zero_division=0),
    })

validation_threshold_results = pd.DataFrame(threshold_rows)
display(validation_threshold_results.round(4))

best_threshold = float(
    validation_threshold_results.loc[
        validation_threshold_results["f1"].idxmax(), "threshold"
    ]
)
print(f"Selected operating threshold (from validation F1): {best_threshold:.2f}")

Selected operating threshold (from validation F1): 0.60


,threshold,predicted_customers,precision,recall,f1
0,0.10,8237,0.0115,1.0000,0.0228
1,0.15,8183,0.0114,0.9789,0.0225
2,0.20,8046,0.0114,0.9684,0.0226
3,0.25,7768,0.0116,0.9474,0.0229
4,0.30,7340,0.0121,0.9368,0.0239
5,0.35,6805,0.0128,0.9158,0.0252
6,0.40,5985,0.0140,0.8842,0.0276
7,0.45,4820,0.0151,0.7684,0.0297
8,0.50,3143,0.0175,0.5789,0.0340
9,0.55,1795,0.0167,0.3158,0.0317


## 6. Final Hold-Out Evaluation on Untouched Test Set

Now we refit candidate models on the full training split (`X_train`, `y_train`) and evaluate once on the held-out `X_test` split, exporting `output/model_metrics.csv`.

In [6]:
fitted_models = {}
metric_rows = []

for model_name, estimator in get_candidate_models().items():
    pipe = Pipeline([
        ("preprocess", make_preprocessor()),
        ("model", estimator),
    ])
    pipe.fit(X_train, y_train)
    scores = pipe.predict_proba(X_test)[:, 1]
    preds = (scores >= best_threshold).astype(int)

    metric_rows.append({
        "model": model_name,
        "selected": model_name == best_name,
        "operating_threshold": best_threshold,
        "average_precision_pr_auc": average_precision_score(y_test, scores),
        "roc_auc": roc_auc_score(y_test, scores),
        "precision_at_threshold": precision_score(y_test, preds, zero_division=0),
        "recall_at_threshold": recall_score(y_test, preds, zero_division=0),
        "f1_at_threshold": f1_score(y_test, preds, zero_division=0),
        "test_positive_rate": float(y_test.mean()),
        "test_rows": int(len(y_test)),
    })
    fitted_models[model_name] = pipe

metrics = (
    pd.DataFrame(metric_rows)
    .sort_values("average_precision_pr_auc", ascending=False)
    .reset_index(drop=True)
)
display(metrics.round(4))
metrics.to_csv(OUTPUT_DIR / "model_metrics.csv", index=False)

best_test_model = fitted_models[best_name]
test_scores = best_test_model.predict_proba(X_test)[:, 1]
test_predictions = (test_scores >= best_threshold).astype(int)

print(f"=== Hold-Out Test Summary ({best_name} @ threshold={best_threshold:.2f}) ===")
print(f"Flagged Customers : {int(test_predictions.sum()):,} / {len(y_test):,}")
print(f"Precision         : {precision_score(y_test, test_predictions, zero_division=0):.4f}")
print(f"Recall            : {recall_score(y_test, test_predictions, zero_division=0):.4f}")
print(f"F1-Score          : {f1_score(y_test, test_predictions, zero_division=0):.4f}")
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, test_predictions))

=== Hold-Out Test Summary (LogisticRegression_balanced @ threshold=0.60) ===
Flagged Customers : 1,053 / 10,403
Precision         : 0.0180
Recall            : 0.1597
F1-Score          : 0.0324

Confusion Matrix:
[[9250 1034]
 [ 100   19]]


,model,selected,operating_threshold,average_precision_pr_auc,roc_auc,precision_at_threshold,recall_at_threshold,f1_at_threshold,test_positive_rate,test_rows
0,LogisticRegression_balanced,True,0.6,0.017,0.5833,0.018,0.1597,0.0324,0.0114,10403
1,RandomForest_balanced,False,0.6,0.016,0.5701,0.000,0.0000,0.0000,0.0114,10403


## 7. Top-K Ranking Lift & Model Diagnostic Curves

Evaluate ranking concentration in the top `1%`, `5%`, `10%`, and `20%` highest-scored customers on `X_test`, and export `output/top_k_lift_metrics.csv`.

In [7]:
ranked_test = pd.DataFrame({
    "actual": y_test.to_numpy(),
    "score": test_scores,
})
baseline_rate = float(ranked_test["actual"].mean())

top_k_rows = []
for fraction in [0.01, 0.05, 0.10, 0.20]:
    n = max(1, int(np.ceil(len(ranked_test) * fraction)))
    top = ranked_test.nlargest(n, "score")
    group_rate = float(top["actual"].mean())
    top_k_rows.append({
        "top_fraction": fraction,
        "customers_in_group": n,
        "actual_repeat_customers": int(top["actual"].sum()),
        "repeat_rate_in_group": group_rate,
        "overall_repeat_rate": baseline_rate,
        "lift_vs_baseline": group_rate / baseline_rate if baseline_rate > 0 else np.nan,
    })

top_k_metrics = pd.DataFrame(top_k_rows)
display(top_k_metrics.round(4))
top_k_metrics.to_csv(OUTPUT_DIR / "top_k_lift_metrics.csv", index=False)

# Plot Precision-Recall, ROC, and Top-K Lift charts
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

for model_name, pipe in fitted_models.items():
    m_scores = pipe.predict_proba(X_test)[:, 1]
    prec, rec, _ = precision_recall_curve(y_test, m_scores)
    ap = average_precision_score(y_test, m_scores)
    axes[0].plot(rec, prec, label=f"{model_name} (AP={ap:.4f})")

    fpr, tpr, _ = roc_curve(y_test, m_scores)
    auc_val = roc_auc_score(y_test, m_scores)
    axes[1].plot(fpr, tpr, label=f"{model_name} (AUC={auc_val:.4f})")

axes[0].axhline(baseline_rate, color="gray", linestyle="--", label=f"Baseline ({baseline_rate:.4f})")
axes[0].set_title("Precision-Recall Curve (Test)")
axes[0].set_xlabel("Recall")
axes[0].set_ylabel("Precision")
axes[0].legend(fontsize=8)

axes[1].plot([0, 1], [0, 1], color="gray", linestyle="--")
axes[1].set_title("ROC Curve (Test)")
axes[1].set_xlabel("False Positive Rate")
axes[1].set_ylabel("True Positive Rate")
axes[1].legend(fontsize=8)

labels_k = [f"Top {int(f*100)}%" for f in top_k_metrics["top_fraction"]]
axes[2].bar(labels_k, top_k_metrics["lift_vs_baseline"], color="#2b5c8f")
axes[2].axhline(1.0, color="gray", linestyle="--", label="Baseline (1.0x)")
axes[2].set_title(f"Top-K Lift ({best_name})")
axes[2].set_ylabel("Lift vs. Baseline")
axes[2].legend(fontsize=8)

plt.tight_layout()
plt.savefig(FIG_DIR / "15_ml_model_evaluation.png", dpi=150)
plt.show()

has vm saved


,top_fraction,customers_in_group,actual_repeat_customers,repeat_rate_in_group,overall_repeat_rate,lift_vs_baseline
0,0.01,105,4,0.0381,0.0114,3.3303
1,0.05,521,11,0.0211,0.0114,1.8457
2,0.10,1041,18,0.0173,0.0114,1.5116
3,0.20,2081,30,0.0144,0.0114,1.2603


## 8. Fit Final Model on Full Snapshot & Export Dashboard Scores

After completing the hold-out evaluation, we fit the selected pipeline on all `52,013` eligible customers to generate dashboard-ready propensity scores (`output/customer_propensity_scores.csv`) and serialize the trained pipeline (`output/repeat_purchase_model.joblib`).

In [8]:
final_estimator = get_candidate_models()[best_name]
final_model = Pipeline([
    ("preprocess", make_preprocessor()),
    ("model", final_estimator),
])
final_model.fit(X, y)
all_scores = final_model.predict_proba(X)[:, 1]

scored = df[[ID_COL, "customer_state", "customer_city", "snapshot_date", "label_window_end"]].copy()
scored["predicted_repeat_probability"] = all_scores
scored["propensity_percentile"] = scored["predicted_repeat_probability"].rank(pct=True)
scored["propensity_segment"] = pd.cut(
    scored["propensity_percentile"],
    bins=[0.0, 0.80, 0.95, 1.0000001],
    labels=["Low (bottom 80%)", "Medium (next 15%)", "High (top 5%)"],
    include_lowest=True,
).astype(str)
scored = scored.sort_values("predicted_repeat_probability", ascending=False).reset_index(drop=True)

scored.to_csv(OUTPUT_DIR / "customer_propensity_scores.csv", index=False)
joblib.dump(final_model, OUTPUT_DIR / "repeat_purchase_model.joblib")

# Verify reloaded model consistency
reloaded_model = joblib.load(OUTPUT_DIR / "repeat_purchase_model.joblib")
assert np.allclose(all_scores, reloaded_model.predict_proba(X)[:, 1]), "Reloaded model mismatch!"

print("Successfully exported:")
print(" - model_artifacts/model_metrics.csv")
print(" - model_artifacts/top_k_lift_metrics.csv")
print(" - model_artifacts/customer_propensity_scores.csv")
print(" - model_artifacts/repeat_purchase_model.joblib")
print(" - model_artifacts/figures/15_ml_model_evaluation.png")
print("\nSegment distribution:")
print(scored["propensity_segment"].value_counts())
display(scored.head(10))


Successfully exported:
 - output/model_metrics.csv
 - output/top_k_lift_metrics.csv
 - output/customer_propensity_scores.csv
 - output/repeat_purchase_model.joblib
 - output/figures/15_ml_model_evaluation.png

Segment distribution:
propensity_segment
Low (bottom 80%)     41610
Medium (next 15%)     7802
High (top 5%)         2601
Name: count, dtype: int64


,customer_unique_id,customer_state,customer_city,snapshot_date,label_window_end,predicted_repeat_probability,propensity_percentile,propensity_segment
0,4546caea018ad8c692964e3382debd19,SP,sao paulo,2018-03-01,2018-08-28,0.997651,1.000000,High (top 5%)
1,b8b3c435a58aebd788a477bed8342910,RS,arroio do sal,2018-03-01,2018-08-28,0.996627,0.999981,High (top 5%)
2,dc813062e0fc23409cd255f7f53c7074,PE,garanhuns,2018-03-01,2018-08-28,0.994849,0.999962,High (top 5%)
3,b39dde6dd619943190a8cc9aa6db38a4,SP,sao bernardo do campo,2018-03-01,2018-08-28,0.994743,0.999942,High (top 5%)
4,63cfc61cee11cbe306bff5857d00bfe4,RJ,rio de janeiro,2018-03-01,2018-08-28,0.993337,0.999923,High (top 5%)
5,1b6c7548a2a1f9037c1fd3ddfed95f33,MG,ituiutaba,2018-03-01,2018-08-28,0.988950,0.999904,High (top 5%)
6,1f98e3467d2b48dfb44fc12b137d3447,SP,sao paulo,2018-03-01,2018-08-28,0.987197,0.999885,High (top 5%)
7,8d50f5eadf50201ccdcedfb9e2ac8455,SP,sao paulo,2018-03-01,2018-08-28,0.987148,0.999865,High (top 5%)
8,11f97da02237a49c8e783dfda6f50e8e,GO,goiania,2018-03-01,2018-08-28,0.987013,0.999846,High (top 5%)
9,3e43e6105506432c953e165fb2acf44c,SP,praia grande,2018-03-01,2018-08-28,0.982872,0.999827,High (top 5%)


## 9. Summary & Recommended Next Steps

- **Selected Baseline:** `LogisticRegression_balanced` (selected strictly via validation PR-AUC; achieves **0.5833 ROC-AUC**, **3.33x lift** in the top 1% of scored customers, and **1.85x lift** in the top 5% on the untouched test split).
- **Recommended Next Steps Before Campaign Execution:**
  1. Perform rolling-origin temporal validation across multiple historical cutoff dates.
  2. Calibrate offer thresholds using unit economics (margin, discount cost, expected order value).
  3. Run a randomized controlled holdout experiment or uplift model before sending discounts.